# 03 - Model Experiments

**Dynamic Pricing Engine** - Model Selection & Training

This notebook covers:
1. Demand forecasting with XGBoost (TimeSeriesSplit CV)
2. Price elasticity estimation with Ridge Regression
3. Revenue optimization with scipy
4. Anomaly detection with Isolation Forest
5. SHAP explainability
6. Model comparison & selection

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.utils.config import load_config, PROJECT_ROOT

config = load_config()
processed_dir = PROJECT_ROOT / config['data']['processed_dir']

## 1. Load Feature-Engineered Data

In [ ]:
listings = pd.read_parquet(processed_dir / 'listings_features.parquet')
calendar = pd.read_parquet(processed_dir / 'calendar_features.parquet')

print(f'Listings: {listings.shape}')
print(f'Calendar: {calendar.shape}')

## 2. Demand Forecasting (XGBoost)

In [ ]:
from src.models.demand_forecaster import DemandForecaster

forecaster = DemandForecaster(config)

# Train with default hyperparameters first
metrics = forecaster.train(calendar, listings, tune_hyperparams=False)
print('\nDemand Forecaster Metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

In [ ]:
# Feature importance
importance = forecaster.get_feature_importance()
print('Top 10 Features for Demand Prediction:')
print(importance.head(10).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
importance.head(15).plot.barh(x='feature', y='importance', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance - Demand Forecaster')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 3. Price Elasticity Estimation

In [ ]:
from src.models.elasticity_estimator import ElasticityEstimator

estimator = ElasticityEstimator(config)
elasticity_metrics = estimator.train(calendar, listings)

print('\nElasticity Estimator Metrics:')
for k, v in elasticity_metrics.items():
    print(f'  {k}: {v:.4f}')

print(f'\nElasticity Coefficient: {estimator.elasticity_coeff:.4f}')
print('Interpretation: A 1% increase in price leads to a '
      f'{abs(estimator.elasticity_coeff):.2f}% decrease in demand')

In [ ]:
# Model coefficients
coefs = estimator.get_coefficients()
print('\nElasticity Model Coefficients:')
print(coefs.to_string(index=False))

## 4. Revenue Optimization

In [ ]:
from src.models.optimizer import get_optimal_price, compute_revenue_curve

# Example optimization for a sample listing
sample_demand = 0.65  # 65% occupancy at current price
sample_price = 150.0  # Current market price
elasticity = estimator.get_elasticity()

result = get_optimal_price(
    base_demand=sample_demand,
    base_price=sample_price,
    elasticity=elasticity,
    config=config,
)

print('Optimization Result:')
print(f'  Base Price:     ${result.base_price}')
print(f'  Optimal Price:  ${result.optimal_price}')
print(f'  Price Range:    ${result.price_range[0]} - ${result.price_range[1]}')
print(f'  Expected Demand: {result.expected_demand:.2%}')
print(f'  Expected Revenue: ${result.expected_revenue}')
print(f'  Revenue Lift:    {result.revenue_lift_pct:+.1f}%')

In [ ]:
# Price vs Revenue curve
curve = compute_revenue_curve(
    base_demand=sample_demand,
    base_price=sample_price,
    elasticity=elasticity,
    config=config,
)

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(x=curve['prices'], y=curve['revenues'],
               name='Revenue', line=dict(color='blue', width=2)),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(x=curve['prices'], y=curve['demands'],
               name='Demand', line=dict(color='green', width=2, dash='dash')),
    secondary_y=True,
)

# Mark optimal price
fig.add_vline(x=result.optimal_price, line_dash='dot', line_color='red',
              annotation_text=f'Optimal: ${result.optimal_price}')
fig.add_vline(x=result.base_price, line_dash='dot', line_color='gray',
              annotation_text=f'Market: ${result.base_price}')

fig.update_layout(
    title='Price vs Revenue Optimization Curve',
    xaxis_title='Price ($)',
    height=500,
)
fig.update_yaxes(title_text='Revenue ($)', secondary_y=False)
fig.update_yaxes(title_text='Demand (Occupancy Rate)', secondary_y=True)
fig.show()

In [ ]:
# Test optimization for 5 different listing profiles
profiles = [
    {'name': 'Budget Private Room', 'price': 60, 'demand': 0.8, 'elasticity': -1.8},
    {'name': 'Mid-Range Apartment', 'price': 150, 'demand': 0.65, 'elasticity': elasticity},
    {'name': 'Luxury Penthouse', 'price': 400, 'demand': 0.4, 'elasticity': -0.8},
    {'name': 'Holiday Hot Spot', 'price': 200, 'demand': 0.9, 'elasticity': -0.5},
    {'name': 'Low-Demand Area', 'price': 100, 'demand': 0.3, 'elasticity': -2.0},
]

print(f'{"Profile":<25} {"Base $":>8} {"Optimal $":>10} {"Lift":>8}')
print('-' * 55)
for p in profiles:
    r = get_optimal_price(p['demand'], p['price'], p['elasticity'], config=config)
    print(f'{p["name"]:<25} ${p["price"]:>6.0f}   ${r.optimal_price:>7.2f}   {r.revenue_lift_pct:>+6.1f}%')

## 5. Anomaly Detection

In [ ]:
from src.models.anomaly_detector import AnomalyDetector

detector = AnomalyDetector(contamination=0.05, config=config)
detector.fit(calendar)

anomalies = detector.predict(calendar)
print(f'Total anomalies: {anomalies.sum():,} / {len(anomalies):,} ({anomalies.mean()*100:.1f}%)')

## 6. SHAP Explainability

In [ ]:
try:
    import shap
    
    # Prepare a sample for SHAP
    feature_names = forecaster.feature_names
    available = [f for f in feature_names if f in calendar.columns]
    X_sample = calendar[available].fillna(0).sample(500, random_state=42)
    
    explainer = shap.TreeExplainer(forecaster.model)
    shap_values = explainer.shap_values(X_sample)
    
    # Global feature importance (SHAP)
    shap.summary_plot(shap_values, X_sample, show=False)
    plt.title('SHAP Feature Importance - Demand Forecaster')
    plt.tight_layout()
    plt.show()
    
    # Single prediction waterfall
    shap.plots.waterfall(shap.Explanation(
        values=shap_values[0],
        base_values=explainer.expected_value,
        data=X_sample.iloc[0],
        feature_names=available,
    ))
    
except ImportError:
    print('SHAP not installed. Run: pip install shap')
except Exception as e:
    print(f'SHAP analysis error: {e}')

## 7. Save Models

In [ ]:
# Save all models for API serving
forecaster.save()
estimator.save()
detector.save()
print('All models saved to models/ directory')

## 8. Model Comparison Summary

| Model | Algorithm | Target | Key Metric | Value |
|-------|-----------|--------|------------|----- |
| Demand Forecaster | XGBoost Classifier | was_booked (0/1) | AUC | _fill after run_ |
| Elasticity Estimator | Ridge Regression | log(demand) | R2 | _fill after run_ |
| Revenue Optimizer | scipy.optimize | Revenue = P*D(P) | Revenue Lift | _fill after run_ |
| Anomaly Detector | Isolation Forest | anomaly flag | Contamination | 5% |

### Key Decisions
- **TimeSeriesSplit CV** used (not random KFold) to prevent data leakage
- **Log-log regression** for elasticity follows economic theory
- **Bounded optimization** prevents nonsensical price recommendations
- **Isolation Forest** flags unusual demand for manual review